# Chapter 6.7: Embedding & Reward Model Serving

## Learning Objectives

By the end of this notebook, you will:

1. **Understand embedding model serving** architecture and requirements
2. **Implement efficient batch processing** for embedding workloads
3. **Learn reward model serving** for RLHF pipelines
4. **Explore classifier model serving** in SGLang
5. **Design APIs** for non-generative model endpoints
6. **Optimize performance** for high-throughput embedding workloads
7. **Walk through SGLang source code** for embedding/reward support
8. **Build a semantic search pipeline** as a practical exercise

## Prerequisites

- Understanding of transformer architecture
- Familiarity with embeddings and vector similarity
- Basic knowledge of RLHF (Reinforcement Learning from Human Feedback)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part6/chapter_6.7_embedding_reward.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part6/chapter_6.7_embedding_reward.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Embedding Models vs Generative Models

### 1.1 Key Differences

```
Generative Model (LLM):                  Embedding Model:
  Input:  "Hello, how"                     Input:  "Search query or document"
  Output: "are you?" (token by token)      Output: [0.23, -0.15, 0.87, ...]
                                                    (fixed-size vector)
  
  Process: Autoregressive decode            Process: Single forward pass
  Cost: O(seq_len * output_len)             Cost: O(seq_len) -- much cheaper!
  Latency: 100ms - 10s                     Latency: 5-50ms
  Batching: Complex (continuous batching)   Batching: Simple (all same phase)
```

### 1.2 Why Serve Embeddings with SGLang?

1. **Unified infrastructure**: Same serving framework for LLMs and embeddings
2. **GPU efficiency**: Maximize GPU utilization with batching
3. **OpenAI-compatible API**: Drop-in replacement for `/v1/embeddings`
4. **Advanced features**: Prefix caching can benefit repeated prefixes

### 1.3 Popular Embedding Models

| Model | Dimensions | Max Tokens | Use Case |
|-------|-----------|-----------|----------|
| E5-large-v2 | 1024 | 512 | General embeddings |
| BGE-large | 1024 | 512 | Multilingual |
| GTE-large | 1024 | 8192 | Long documents |
| Nomic-embed-text | 768 | 8192 | Efficient embeddings |
| Jina-embeddings-v3 | 1024 | 8192 | Flexible dimensions |

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import time
from typing import List, Dict

# Simulate embedding model serving

class SimpleEmbeddingModel(nn.Module):
    """Simplified embedding model for demonstration."""
    
    def __init__(self, vocab_size=10000, hidden_dim=256, n_layers=4, max_seq_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.position_embedding = nn.Embedding(max_seq_len, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=8, dim_feedforward=hidden_dim*4,
            batch_first=True, dropout=0.0
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.hidden_dim = hidden_dim
    
    def forward(self, input_ids, attention_mask=None):
        """
        Compute embeddings for input tokens.
        Returns mean-pooled hidden states as the embedding.
        """
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        
        h = self.embedding(input_ids) + self.position_embedding(positions)
        h = self.encoder(h, src_key_padding_mask=~attention_mask if attention_mask is not None else None)
        
        # Mean pooling over non-padded tokens
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            h = (h * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            h = h.mean(dim=1)
        
        # L2 normalize
        h = h / h.norm(dim=-1, keepdim=True)
        return h

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SimpleEmbeddingModel().to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Embedding model: {n_params:,} parameters ({n_params/1e6:.1f}M)")
print(f"Output dimension: {model.hidden_dim}")
print(f"Device: {device}")

In [ ]:
# Simple tokenizer simulation
import hashlib

class SimpleTokenizer:
    """Simple word-level tokenizer for demonstration."""
    
    def __init__(self, vocab_size=10000):
        self.vocab_size = vocab_size
    
    def encode(self, text, max_length=128):
        """Convert text to token IDs."""
        words = text.lower().split()
        tokens = [hash(w) % (self.vocab_size - 2) + 1 for w in words]  # Reserve 0 for padding
        tokens = tokens[:max_length]
        return tokens
    
    def batch_encode(self, texts, max_length=128):
        """Encode multiple texts with padding."""
        all_tokens = [self.encode(t, max_length) for t in texts]
        max_len = max(len(t) for t in all_tokens)
        
        # Pad to max length
        padded = []
        masks = []
        for tokens in all_tokens:
            pad_len = max_len - len(tokens)
            padded.append(tokens + [0] * pad_len)
            masks.append([True] * len(tokens) + [False] * pad_len)
        
        return {
            "input_ids": torch.tensor(padded, dtype=torch.long),
            "attention_mask": torch.tensor(masks, dtype=torch.bool),
        }

tokenizer = SimpleTokenizer()

# Test embedding computation
texts = [
    "The quick brown fox jumps over the lazy dog",
    "A fast red fox leaps above the sleepy hound",
    "Machine learning is a subset of artificial intelligence",
    "Deep neural networks learn hierarchical representations",
]

encoded = tokenizer.batch_encode(texts)
input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

with torch.no_grad():
    embeddings = model(input_ids, attention_mask)

print(f"Input texts: {len(texts)}")
print(f"Token shape: {input_ids.shape}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding norm (should be ~1.0): {embeddings.norm(dim=-1)}")

# Compute similarity matrix
similarity = embeddings @ embeddings.T
print(f"\nSimilarity matrix:")
for i, t in enumerate(texts):
    print(f"  {t[:40]:<42s}", end="")
    for j in range(len(texts)):
        print(f" {similarity[i,j].item():6.3f}", end="")
    print()

## 2. Batch Processing for Embeddings

### 2.1 Why Batching Matters

Unlike generative models, embedding computation is a single forward pass. This makes batching straightforward and highly beneficial:

```
No batching:          With batching:
Req 1 -> forward      [Req 1, 2, 3, 4] -> forward
Req 2 -> forward      1 forward pass for 4 requests!
Req 3 -> forward      
Req 4 -> forward      Time: ~1x (GPU utilization much higher)
Time: 4x
```

### 2.2 Batching Strategies

| Strategy | Description | Trade-off |
|----------|-------------|----------|
| Fixed batch | Wait for N requests | Simple but variable latency |
| Time-based | Wait max T ms | Bounded latency, variable batch |
| Dynamic | Hybrid N + T | Best of both worlds |
| Continuous | Always process available | Lowest latency, complex |

In [ ]:
# Embedding batch processor

class EmbeddingBatchProcessor:
    """Efficient batch processing for embedding requests."""
    
    def __init__(self, model, tokenizer, max_batch_size=64, max_wait_ms=10):
        self.model = model
        self.tokenizer = tokenizer
        self.max_batch_size = max_batch_size
        self.max_wait_ms = max_wait_ms
        self.queue = []
        self.stats = {
            "total_requests": 0,
            "total_batches": 0,
            "total_tokens": 0,
            "total_time_ms": 0,
        }
    
    def process_batch(self, texts: List[str]) -> np.ndarray:
        """Process a batch of texts and return embeddings."""
        start = time.perf_counter()
        
        # Tokenize
        encoded = self.tokenizer.batch_encode(texts)
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        
        # Forward pass
        with torch.no_grad():
            embeddings = self.model(input_ids, attention_mask)
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        elapsed = (time.perf_counter() - start) * 1000
        
        # Update stats
        self.stats["total_requests"] += len(texts)
        self.stats["total_batches"] += 1
        self.stats["total_tokens"] += input_ids.numel()
        self.stats["total_time_ms"] += elapsed
        
        return embeddings.cpu().numpy()
    
    def benchmark(self, texts: List[str], batch_sizes: List[int]):
        """Benchmark different batch sizes."""
        results = {}
        
        for bs in batch_sizes:
            # Process all texts in batches of size bs
            n_batches = (len(texts) + bs - 1) // bs
            total_time = 0
            
            for i in range(n_batches):
                batch = texts[i*bs : (i+1)*bs]
                start = time.perf_counter()
                _ = self.process_batch(batch)
                total_time += (time.perf_counter() - start) * 1000
            
            throughput = len(texts) / (total_time / 1000)
            avg_latency = total_time / n_batches
            results[bs] = {
                "total_time_ms": total_time,
                "throughput_rps": throughput,
                "avg_batch_latency_ms": avg_latency,
                "n_batches": n_batches,
            }
        
        return results

# Generate test data
test_texts = [
    f"This is test document number {i} about topic {i % 10}" 
    for i in range(200)
]

processor = EmbeddingBatchProcessor(model, tokenizer)
batch_sizes = [1, 4, 8, 16, 32, 64]

results = processor.benchmark(test_texts, batch_sizes)

print("=== Embedding Batch Size Benchmark ===")
print(f"Total texts: {len(test_texts)}\n")
print(f"{'Batch Size':>10s} | {'Total (ms)':>12s} | {'Throughput':>12s} | {'Latency/batch':>14s} | {'N Batches':>10s}")
print("-" * 65)
for bs, r in results.items():
    print(f"{bs:>10d} | {r['total_time_ms']:>11.1f} | {r['throughput_rps']:>10.0f}/s | "
          f"{r['avg_batch_latency_ms']:>13.1f} | {r['n_batches']:>10d}")

## 3. Reward Model Serving

### 3.1 What is a Reward Model?

A reward model scores how good a response is, used in RLHF pipelines:

```
Input:  (prompt, response) pair
Output: scalar reward score (higher = better)

Architecture:
  [Prompt + Response] -> [Transformer Encoder] -> [Reward Head] -> score
  
  The reward head is typically:
  - A linear layer on top of the last hidden state
  - Maps hidden_dim -> 1 (scalar reward)
```

### 3.2 Use Cases

1. **Online RLHF**: Score model outputs during PPO/DPO training
2. **Best-of-N sampling**: Generate N responses, pick highest reward
3. **Quality filtering**: Filter generated content by reward score
4. **Preference ranking**: Rank multiple responses

### 3.3 Serving Requirements

| Requirement | Embedding Model | Reward Model | Generative LLM |
|------------|-----------------|--------------|----------------|
| Output | Vector | Scalar | Token sequence |
| Forward passes | 1 | 1 | N (autoregressive) |
| KV-cache needed | No | No | Yes |
| Batching | Simple | Simple | Complex |
| Typical latency | 5-20ms | 10-50ms | 100ms-10s |

In [ ]:
# Reward model implementation

class SimpleRewardModel(nn.Module):
    """Simplified reward model for demonstration."""
    
    def __init__(self, vocab_size=10000, hidden_dim=256, n_layers=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.position_embedding = nn.Embedding(512, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=8, dim_feedforward=hidden_dim*4,
            batch_first=True, dropout=0.0
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # Reward head: maps hidden states to scalar score
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),  # Scalar output
        )
    
    def forward(self, input_ids, attention_mask=None):
        """Compute reward score for input."""
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        
        h = self.embedding(input_ids) + self.position_embedding(positions)
        h = self.encoder(h, src_key_padding_mask=~attention_mask if attention_mask is not None else None)
        
        # Use last token's hidden state (like GPT-style)
        if attention_mask is not None:
            # Find the last non-padded position
            last_pos = attention_mask.sum(dim=1) - 1  # [batch]
            h_last = h[torch.arange(h.shape[0]), last_pos]  # [batch, hidden]
        else:
            h_last = h[:, -1, :]  # Last position
        
        # Compute reward score
        reward = self.reward_head(h_last).squeeze(-1)  # [batch]
        return reward

# Create reward model
reward_model = SimpleRewardModel().to(device).eval()
print(f"Reward model parameters: {sum(p.numel() for p in reward_model.parameters()):,}")

# Test with prompt-response pairs
prompts_responses = [
    "Question: What is Python? Answer: Python is a programming language known for readability",
    "Question: What is Python? Answer: Python is a snake found in tropical regions",
    "Question: What is Python? Answer: I don't know random words here",
    "Question: Explain gravity Answer: Gravity is a fundamental force that attracts objects with mass",
]

encoded = tokenizer.batch_encode(prompts_responses)
input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

with torch.no_grad():
    rewards = reward_model(input_ids, attention_mask)

print("\n=== Reward Scores ===")
for text, reward in zip(prompts_responses, rewards):
    print(f"  {text[:60]:<62s} -> reward: {reward.item():.4f}")

## 4. Classifier Model Serving

### 4.1 Classification Head

Similar to reward models, but outputs class probabilities:

```python
class ClassifierModel(nn.Module):
    def __init__(self, base_model, n_classes):
        self.base = base_model  # Transformer encoder
        self.classifier = nn.Linear(hidden_dim, n_classes)
    
    def forward(self, input_ids):
        h = self.base(input_ids)  # [batch, seq, hidden]
        h_cls = h[:, 0, :]        # CLS token representation
        logits = self.classifier(h_cls)  # [batch, n_classes]
        return logits
```

### 4.2 Use Cases

- **Content moderation**: Classify text as safe/unsafe
- **Sentiment analysis**: Positive/negative/neutral
- **Topic classification**: Route requests by topic
- **Language detection**: Identify input language

In [ ]:
# Classifier model for content moderation

class ContentModerationModel(nn.Module):
    """Simplified content moderation classifier."""
    
    LABELS = ["safe", "harassment", "hate_speech", "violence", "sexual", "self_harm"]
    
    def __init__(self, vocab_size=10000, hidden_dim=256, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.position_embedding = nn.Embedding(512, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=8, dim_feedforward=hidden_dim*2,
            batch_first=True, dropout=0.0
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Linear(hidden_dim, len(self.LABELS))
    
    def forward(self, input_ids, attention_mask=None):
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        h = self.embedding(input_ids) + self.position_embedding(positions)
        h = self.encoder(h)
        # Mean pooling
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            h = (h * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            h = h.mean(dim=1)
        logits = self.classifier(h)
        return logits
    
    def predict(self, logits):
        probs = torch.softmax(logits, dim=-1)
        predicted_class = probs.argmax(dim=-1)
        return [
            {
                "label": self.LABELS[c.item()],
                "score": probs[i, c].item(),
                "all_scores": {l: probs[i, j].item() for j, l in enumerate(self.LABELS)}
            }
            for i, c in enumerate(predicted_class)
        ]

mod_model = ContentModerationModel().to(device).eval()

test_texts = [
    "Have a wonderful day everyone",
    "The weather is nice today",
    "I want to learn about cooking recipes",
    "Tell me about machine learning algorithms",
]

encoded = tokenizer.batch_encode(test_texts)
with torch.no_grad():
    logits = mod_model(encoded["input_ids"].to(device), encoded["attention_mask"].to(device))
    predictions = mod_model.predict(logits)

print("=== Content Moderation Results ===")
for text, pred in zip(test_texts, predictions):
    print(f"\n  Text: '{text}'")
    print(f"  Prediction: {pred['label']} (confidence: {pred['score']:.2%})")

## 5. API Design for Non-Generative Models

### 5.1 OpenAI-Compatible Embedding API

```python
# POST /v1/embeddings
{
    "model": "text-embedding-model",
    "input": ["text to embed", "another text"],
    "encoding_format": "float"  # or "base64"
}

# Response:
{
    "object": "list",
    "data": [
        {
            "object": "embedding",
            "embedding": [0.23, -0.15, 0.87, ...],
            "index": 0
        },
        {
            "object": "embedding",
            "embedding": [0.11, 0.42, -0.33, ...],
            "index": 1
        }
    ],
    "model": "text-embedding-model",
    "usage": {"prompt_tokens": 12, "total_tokens": 12}
}
```

### 5.2 SGLang Embedding Server

```bash
# Launch embedding model server
python -m sglang.launch_server \
    --model BAAI/bge-large-en-v1.5 \
    --is-embedding \
    --port 30000

# Or for reward model:
python -m sglang.launch_server \
    --model OpenAssistant/reward-model-deberta-v3-large \
    --is-embedding \
    --port 30000
```

### 5.3 Source Code: Embedding Support

```python
# From sglang/srt/managers/scheduler.py

class Scheduler:
    def process_embedding_request(self, request):
        """
        Handle embedding requests differently from generation:
        - No KV-cache needed
        - No autoregressive loop
        - Simple batch forward pass
        """
        # Tokenize
        input_ids = self.tokenizer.encode(request.text)
        
        # Add to batch (embeddings can be batched efficiently)
        self.embedding_batch.append({
            "request_id": request.id,
            "input_ids": input_ids,
        })
        
        # Process batch when full or timeout
        if self.should_flush_embedding_batch():
            self.run_embedding_batch()
```

In [ ]:
# Simulated embedding API server

class EmbeddingServer:
    """Simulated embedding API server with batching."""
    
    def __init__(self, model, tokenizer, max_batch_size=32, max_wait_ms=5):
        self.model = model
        self.tokenizer = tokenizer
        self.max_batch_size = max_batch_size
        self.max_wait_ms = max_wait_ms
        self.request_queue = []
        self.stats = {"total_requests": 0, "total_batches": 0, "total_time_ms": 0}
    
    def embed(self, texts: List[str]) -> Dict:
        """OpenAI-compatible embedding API."""
        start = time.perf_counter()
        
        # Tokenize
        encoded = self.tokenizer.batch_encode(texts)
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        
        # Compute embeddings
        with torch.no_grad():
            embeddings = self.model(input_ids, attention_mask)
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        elapsed = (time.perf_counter() - start) * 1000
        self.stats["total_requests"] += len(texts)
        self.stats["total_batches"] += 1
        self.stats["total_time_ms"] += elapsed
        
        # Format response
        emb_list = embeddings.cpu().numpy()
        
        response = {
            "object": "list",
            "data": [
                {
                    "object": "embedding",
                    "embedding": emb_list[i].tolist(),
                    "index": i,
                }
                for i in range(len(texts))
            ],
            "model": "simple-embedding-model",
            "usage": {
                "prompt_tokens": int(input_ids.numel()),
                "total_tokens": int(input_ids.numel()),
            }
        }
        
        return response
    
    def get_stats(self):
        avg_latency = self.stats["total_time_ms"] / max(self.stats["total_batches"], 1)
        throughput = self.stats["total_requests"] / max(self.stats["total_time_ms"] / 1000, 0.001)
        return {
            **self.stats,
            "avg_latency_ms": avg_latency,
            "throughput_rps": throughput,
        }

# Test the API
server = EmbeddingServer(model, tokenizer)

response = server.embed(["Hello world", "Goodbye world"])

print("=== Embedding API Response ===")
print(f"Object: {response['object']}")
print(f"Embeddings: {len(response['data'])}")
print(f"Dimension: {len(response['data'][0]['embedding'])}")
print(f"Usage: {response['usage']}")
print(f"\nFirst 5 values of embedding 0: {response['data'][0]['embedding'][:5]}")

## 6. Performance Optimization

### 6.1 Optimization Strategies

1. **Dynamic batching**: Accumulate requests and process together
2. **Sequence length sorting**: Group similar-length sequences to minimize padding
3. **Prefix caching**: For repeated prefixes (e.g., instruction templates)
4. **Quantization**: FP16 -> INT8 for faster inference
5. **Model sharding**: Tensor parallel for large embedding models

In [ ]:
# Sequence length sorting optimization

import random

def generate_varied_texts(n=100):
    """Generate texts with varying lengths."""
    templates = [
        "short text",
        "this is a medium length text with several words in it",
        "this is a longer text that contains many more words and provides more context " * 3,
        "very " * 50 + "long text",
    ]
    return [random.choice(templates) + f" {i}" for i in range(n)]

random.seed(42)
varied_texts = generate_varied_texts(200)

# Without sorting: process as-is
proc = EmbeddingBatchProcessor(model, tokenizer)
start = time.perf_counter()
for i in range(0, len(varied_texts), 32):
    batch = varied_texts[i:i+32]
    _ = proc.process_batch(batch)
unsorted_time = (time.perf_counter() - start) * 1000

# With sorting: group by length
indexed = [(i, t) for i, t in enumerate(varied_texts)]
sorted_texts = sorted(indexed, key=lambda x: len(x[1].split()))

start = time.perf_counter()
for i in range(0, len(sorted_texts), 32):
    batch = [t[1] for t in sorted_texts[i:i+32]]
    _ = proc.process_batch(batch)
sorted_time = (time.perf_counter() - start) * 1000

print("=== Sequence Length Sorting Optimization ===")
print(f"Unsorted: {unsorted_time:.1f} ms")
print(f"Sorted:   {sorted_time:.1f} ms")
print(f"Speedup:  {unsorted_time/sorted_time:.2f}x")
print(f"\nReason: Sorting reduces padding waste.")
print(f"  When a batch has [5 tokens, 200 tokens], the 5-token text is padded to 200.")
print(f"  Sorting groups similar lengths, reducing wasted computation.")

## 7. Building a Semantic Search Pipeline

### 7.1 Architecture

```
Indexing Phase:
  Documents -> Embedding Model -> Vector Store

Query Phase:
  Query -> Embedding Model -> Vector Similarity Search -> Top-K Results
```

In [ ]:
# Simple semantic search pipeline

class SemanticSearch:
    """Simple semantic search using embedding model."""
    
    def __init__(self, embedding_model, tokenizer):
        self.model = embedding_model
        self.tokenizer = tokenizer
        self.documents = []
        self.embeddings = None
    
    def index(self, documents: List[str]):
        """Index documents by computing their embeddings."""
        self.documents = documents
        
        # Compute embeddings in batches
        all_embeddings = []
        batch_size = 32
        
        for i in range(0, len(documents), batch_size):
            batch = documents[i:i+batch_size]
            encoded = self.tokenizer.batch_encode(batch)
            
            with torch.no_grad():
                embs = self.model(
                    encoded["input_ids"].to(device),
                    encoded["attention_mask"].to(device)
                )
            all_embeddings.append(embs.cpu())
        
        self.embeddings = torch.cat(all_embeddings, dim=0)
        print(f"Indexed {len(documents)} documents. Embedding shape: {self.embeddings.shape}")
    
    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """Search for documents most similar to query."""
        # Compute query embedding
        encoded = self.tokenizer.batch_encode([query])
        with torch.no_grad():
            query_emb = self.model(
                encoded["input_ids"].to(device),
                encoded["attention_mask"].to(device)
            ).cpu()
        
        # Compute cosine similarity
        similarities = (query_emb @ self.embeddings.T).squeeze(0)
        
        # Get top-K
        top_k_indices = similarities.argsort(descending=True)[:top_k]
        
        results = []
        for idx in top_k_indices:
            results.append({
                "document": self.documents[idx],
                "score": similarities[idx].item(),
                "index": idx.item(),
            })
        
        return results

# Build search index
documents = [
    "Python is a high level programming language known for its readability",
    "JavaScript is the language of the web used for frontend development",
    "Machine learning algorithms learn patterns from data",
    "Deep learning uses neural networks with many layers",
    "Natural language processing enables computers to understand text",
    "Computer vision helps machines interpret visual information",
    "Reinforcement learning trains agents through trial and error",
    "Transfer learning reuses pretrained models for new tasks",
    "The solar system contains eight planets orbiting the sun",
    "Photosynthesis converts sunlight into chemical energy in plants",
    "Gravity is the force of attraction between objects with mass",
    "DNA carries genetic information in living organisms",
    "The stock market fluctuates based on supply and demand",
    "Quantum computing uses quantum bits for computation",
    "Climate change affects global weather patterns and ecosystems",
]

search_engine = SemanticSearch(model, tokenizer)
search_engine.index(documents)

# Search
queries = [
    "How do machines learn from data?",
    "What programming language is good for beginners?",
    "How does the universe work?",
]

print("\n=== Semantic Search Results ===")
for query in queries:
    print(f"\nQuery: '{query}'")
    results = search_engine.search(query, top_k=3)
    for i, r in enumerate(results):
        print(f"  {i+1}. [{r['score']:.3f}] {r['document']}")

## 8. Summary

### Key Takeaways

1. **Embedding models** require only a single forward pass (no autoregressive loop), making them much simpler to serve than generative LLMs.

2. **Batch processing** is crucial for embedding throughput -- group requests and process them together for GPU efficiency.

3. **Reward models** output scalar scores and serve RLHF pipelines for quality evaluation, best-of-N sampling, and content filtering.

4. **SGLang provides unified serving** for embeddings, rewards, classifiers, and generative models through the `--is-embedding` flag.

5. **Sequence length sorting** reduces padding waste and improves throughput.

6. **The OpenAI-compatible API** (`/v1/embeddings`) enables drop-in replacement with SGLang.

## Exercises

### Exercise 1: Build a Semantic Search Pipeline

Extend the SemanticSearch class to support:
1. Incremental indexing (add documents without re-indexing everything)
2. Metadata filtering (e.g., search only documents from a specific category)
3. Hybrid search (combine embedding similarity with keyword matching)

### Exercise 2: Best-of-N Sampling with Reward Model

Implement a best-of-N pipeline:
1. Generate N responses for a prompt
2. Score each with the reward model
3. Return the highest-scoring response
4. Measure quality improvement vs N

### Exercise 3: Embedding Server Load Test

Build a load testing tool that:
1. Sends concurrent embedding requests
2. Measures throughput and latency at different concurrency levels
3. Finds the optimal batch size for your GPU

### Exercise 4: Multi-Model Serving

Design a serving system that:
1. Serves both an embedding model and a generative model
2. Uses the embedding model for retrieval-augmented generation (RAG)
3. Measures end-to-end latency for RAG queries

In [ ]:
# Exercise 1 Starter Code: Enhanced Semantic Search

class EnhancedSemanticSearch(SemanticSearch):
    """Enhanced semantic search with incremental indexing and filtering."""
    
    def __init__(self, embedding_model, tokenizer):
        super().__init__(embedding_model, tokenizer)
        self.metadata = []  # List of metadata dicts per document
    
    def add_documents(self, documents: List[str], metadata: List[Dict] = None):
        """
        TODO: Add documents incrementally without re-indexing.
        
        Hints:
        1. Compute embeddings for new documents only
        2. Concatenate with existing embeddings
        3. Append metadata
        """
        # YOUR CODE HERE
        pass
    
    def search_with_filter(self, query: str, top_k: int = 5, 
                           filter_fn=None) -> List[Dict]:
        """
        TODO: Search with metadata filtering.
        
        Args:
            filter_fn: function(metadata) -> bool
                       Only include documents where filter returns True
        """
        # YOUR CODE HERE
        pass

print("Exercise 1: Implement add_documents and search_with_filter methods.")

In [ ]:
# Exercise 2 Starter Code: Best-of-N Sampling

def best_of_n_sampling(prompt, n_samples, generate_fn, reward_fn):
    """
    TODO: Implement best-of-N sampling.
    
    Args:
        prompt: input prompt
        n_samples: number of responses to generate
        generate_fn: function(prompt) -> response_text
        reward_fn: function(prompt + response) -> score
    
    Returns:
        best_response: the response with the highest reward
        best_score: its reward score
        all_scores: list of all scores for analysis
    """
    # YOUR CODE HERE
    pass

print("Exercise 2: Implement best_of_n_sampling.")
print("Goal: Generate N responses, score each, return the best.")